# 1. Check data duplication, ROI, quality of images

## 1.1 Import libraries

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
import imagehash
import torch
from torchvision import models, transforms
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
import umap
import matplotlib.pyplot as plt
import shutil

## 1.2 Configuration of data source folder and clean data folder

In [ ]:
RAW_DATA_DIR = "C:/Users/ekadw/Documents/DATA/Leaf_Disease/pepper"
CLEAN_DATA_DIR = "C:/Users/ekadw/Documents/DATA/Leaf_Disease/clean_pepper_leaves"
os.makedirs(CLEAN_DATA_DIR, exist_ok=True)

## 1.3 Remove duplicates

In [ ]:
def remove_duplicates_recursive(base_dir, hash_size=16, threshold=5):
    print("\n🔍 Step 1: Removing duplicates...")
    total_removed = 0
    for class_folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, class_folder)
        if not os.path.isdir(folder_path):
            continue

        hashes = {}
        removed = 0
        for file in tqdm(os.listdir(folder_path), desc=f"{class_folder}"):
            if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            path = os.path.join(folder_path, file)
            try:
                with Image.open(path) as img:
                    h = imagehash.phash(img, hash_size=hash_size)
            except:
                os.remove(path)
                continue

            duplicate = any(abs(h - ex_hash) < threshold for ex_hash in hashes.values())
            if duplicate:
                os.remove(path)
                removed += 1
            else:
                hashes[file] = h

        total_removed += removed
        print(f"  ↳ Removed {removed} duplicates in {class_folder}")

    print(f"✅ Total duplicates removed: {total_removed}\n")

remove_duplicates_recursive(RAW_DATA_DIR)

## 1.4 ROI extraction

In [ ]:
def extract_leaf_roi(image_path, save_path, min_area=500):
    img = cv2.imread(image_path)
    if img is None:
        return False

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_green = np.array([20, 30, 20])
    upper_green = np.array([100, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest = max(contours, key=cv2.contourArea)
        if cv2.contourArea(largest) >= min_area:
            x, y, w, h = cv2.boundingRect(largest)
            img = img[y:y+h, x:x+w]

    cv2.imwrite(save_path, img)
    return True

print("🍃 Step 2: Extracting ROI regions...")
for root, dirs, files in os.walk(RAW_DATA_DIR):
    for file in tqdm(files, desc=f"Processing {os.path.basename(root)}"):
        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        src_path = os.path.join(root, file)
        rel_path = os.path.relpath(root, RAW_DATA_DIR)
        save_dir = os.path.join(CLEAN_DATA_DIR, rel_path)
        os.makedirs(save_dir, exist_ok=True)
        dst_path = os.path.join(save_dir, file)
        extract_leaf_roi(src_path, dst_path)

print("✅ ROI extraction complete.\n")

## 1.5 Quality of filtering

In [ ]:
def is_blurry(image_path, threshold=30):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True
    lap_var = cv2.Laplacian(img, cv2.CV_64F).var()
    return lap_var < threshold

def remove_low_quality_images_recursive(base_dir, min_size=(64, 64)):
    print("🧹 Step 3: Removing low-quality images...")
    total_removed = 0
    for root, dirs, files in os.walk(base_dir):
        removed = 0
        for file in files:
            path = os.path.join(root, file)
            try:
                img = cv2.imread(path)
                if img is None:
                    os.remove(path)
                    continue
                h, w = img.shape[:2]
                if h < min_size[0] or w < min_size[1] or is_blurry(path):
                    os.remove(path)
                    removed += 1
            except:
                os.remove(path)
                removed += 1
        if removed:
            print(f"  ↳ Removed {removed} low-quality images in {root}")
            total_removed += removed
    print(f"✅ Total low-quality removed: {total_removed}\n")

remove_low_quality_images_recursive(CLEAN_DATA_DIR)

## 1.6 Visual quality of check (PCA +UMAP)

In [ ]:
print("📊 Step 4: Visualizing with PCA + UMAP...")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.resnet50(weights="IMAGENET1K_V2")
model = torch.nn.Sequential(*list(model.children())[:-1])
model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def extract_features(img_path):
    try:
        img = Image.open(img_path).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = model(x).cpu().numpy().flatten()
        return feat
    except:
        return None

features, paths = [], []
for root, dirs, files in os.walk(CLEAN_DATA_DIR):
    for file in files:
        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        path = os.path.join(root, file)
        feat = extract_features(path)
        if feat is not None:
            features.append(feat)
            paths.append(path)

features = np.array(features)
print(f"Extracted {len(features)} feature vectors")

if len(features) > 0:
    pca = PCA(n_components=50).fit_transform(features)
    reducer = umap.UMAP(random_state=42)
    embedding = reducer.fit_transform(pca)

    iso = IsolationForest(contamination=0.03, random_state=42)
    preds = iso.fit_predict(pca)

    outliers = preds == -1
    print(f"Detected {outliers.sum()} outliers out of {len(outliers)}")

    plt.figure(figsize=(10, 8))
    plt.scatter(embedding[~outliers, 0], embedding[~outliers, 1], s=5, label="Normal", alpha=0.5)
    plt.scatter(embedding[outliers, 0], embedding[outliers, 1], s=10, color="red", label="Outliers")
    plt.legend()
    plt.title("UMAP Visualization of Tomato Leaf Dataset")
    plt.show()

    for i, is_outlier in enumerate(outliers):
        if is_outlier:
            os.remove(paths[i])

print("✅ Final dataset ready for CNN training!")

# 2. Plot UMAP best quality of data

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.decomposition import PCA
import umap
import torch
from torchvision import models, transforms

# ===============================
# CONFIGURATION
# ===============================
CLEAN_DATA_DIR = "C:/Users/ekadw/Documents/DATA/Leaf_Disease/clean_pepper_leaves"
device = "cuda" if torch.cuda.is_available() else "cpu"

# ===============================
# STEP 1: LOAD PRETRAINED MODEL (for embeddings)
# ===============================
model = models.resnet50(weights="IMAGENET1K_V2")
model = torch.nn.Sequential(*list(model.children())[:-1])  # remove final FC layer
model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ===============================
# STEP 2: EXTRACT FEATURES
# ===============================
features = []
labels = []
class_names = sorted(os.listdir(CLEAN_DATA_DIR))

for label_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(CLEAN_DATA_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue
    print(f"Extracting features from class: {class_name}")
    for file in tqdm(os.listdir(class_dir), desc=f"{class_name}"):
        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        img_path = os.path.join(class_dir, file)
        try:
            img = Image.open(img_path).convert("RGB")
            x = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = model(x).cpu().numpy().flatten()
            features.append(feat)
            labels.append(label_idx)
        except Exception as e:
            print(f"Error on {img_path}: {e}")

features = np.array(features)
labels = np.array(labels)

print(f"Total samples: {len(labels)}")
print("Feature shape:", features.shape)

# ===============================
# STEP 3: PCA + UMAP REDUCTION
# ===============================
pca = PCA(n_components=50).fit_transform(features)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(pca)

print("UMAP embedding shape:", embedding.shape)

# ===============================
# STEP 4: PLOT CLUSTERS
# ===============================
plt.figure(figsize=(10, 8))
for i, class_name in enumerate(class_names):
    mask = labels == i
    plt.scatter(
        embedding[mask, 0],
        embedding[mask, 1],
        s=10,
        label=class_name,
        alpha=0.7
    )

plt.title("UMAP Clustering of Tomato Leaf Dataset (Post-Cleaning)")
plt.legend()
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.show()


# 3. From clean data to deployment

In [1]:
"""
Tomato Leaf Disease Classification with MobileNetV2 (PyTorch, CPU-optimized)

Steps:
1. Check dataset & target distribution
2. Data augmentation and normalization
3. Train/val/test split
4. Compute class weights
5. Build MobileNetV2 (pretrained)
6. Train base model (frozen)
7. Fine-tune top layers
8. Evaluate model
9. Save model and class indices

Author: Eka Dwipayana (optimized for CPU training)
"""

'\nTomato Leaf Disease Classification with MobileNetV2 (PyTorch, CPU-optimized)\n\nSteps:\n1. Check dataset & target distribution\n2. Data augmentation and normalization\n3. Train/val/test split\n4. Compute class weights\n5. Build MobileNetV2 (pretrained)\n6. Train base model (frozen)\n7. Fine-tune top layers\n8. Evaluate model\n9. Save model and class indices\n\nAuthor: Eka Dwipayana (optimized for CPU training)\n'

## 1. Import libraries

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pandas as pd
import multiprocessing

## 2. Setup

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")
NUM_WORKERS = max(1, multiprocessing.cpu_count() - 1)
BATCH_SIZE = 16
DATA_DIR = "C:/Users/ekadw/Documents/DATA/Leaf_Disease/clean_pepper_leaves"  # <-- change this to your dataset root

print(f"Using {NUM_WORKERS} CPU cores")
print(f"Using device: {device}")

## 3. Explore target distribution

In [ ]:
classes = sorted(os.listdir(DATA_DIR))
counts = {cls: len(os.listdir(os.path.join(DATA_DIR, cls))) for cls in classes}

plt.figure(figsize=(10, 5))
sns.barplot(
    x=list(counts.keys()),
    y=list(counts.values()),
    hue=list(counts.keys()),
    palette="viridis",
    legend=False
)
plt.title("Target Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Data transforms (with augmentation)

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=train_tfms)

train_size = int(0.7 * len(dataset))
val_size   = int(0.2 * len(dataset))
test_size  = len(dataset) - train_size - val_size
train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])
val_ds.dataset.transform = val_tfms
test_ds.dataset.transform = val_tfms

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 5. Class weights for imbalance

In [ ]:
targets = [dataset.samples[i][1] for i in range(len(dataset))]
class_weights = compute_class_weight('balanced', classes=np.unique(targets), y=targets)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class Weights:", class_weights)

## 6. Model setup (MobileNetV2)

In [ ]:
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
for param in model.features.parameters():
    param.requires_grad = False

num_classes = len(classes)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.2)

## 7. Training function (with early stopping)

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=15, patience=5, save_path="best_model.pth"):
    best_acc = 0.0
    best_epoch = 0
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        running_loss, correct, total = 0, 0, 0

        for imgs, labels in tqdm(train_loader):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    print(f"Best Val Acc: {best_acc:.4f} at epoch {best_epoch+1}")
    return history

## 8. Train base model

In [ ]:
history = train_model(model, criterion, optimizer, scheduler, num_epochs=15, patience=5, save_path="mobilenetv2_base.pth")

## 9. Fine-tunning (Unfreeze last 3 layers)

In [ ]:
for param in model.features[-3:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)
history_ft = train_model(model, criterion, optimizer, scheduler, num_epochs=10, patience=4, save_path="mobilenetv2_finetuned.pth")

## 10. Plot training history

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history["train_acc"], label="Train Acc")
plt.plot(history["val_acc"], label="Val Acc")
plt.title("Training vs Validation Accuracy (Base)")
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_ft["train_acc"], label="Train Acc (FT)")
plt.plot(history_ft["val_acc"], label="Val Acc (FT)")
plt.title("Fine-tuning Accuracy")
plt.legend()
plt.show()

## 11. Evaluate on test set

In [ ]:
model.load_state_dict(torch.load("mobilenetv2_finetuned.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=classes))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.show()

## 12. Save model and class indices

In [ ]:
torch.save(model.state_dict(), "mobilenetv2_pepper_best.pth")
class_to_idx = dataset.class_to_idx
with open("class_indices.json", "w") as f:
    json.dump(class_to_idx, f)
print("✅ Model and class indices saved successfully.")

## 13. Create requirements.txt

In [ ]:
reqs = [
    "torch",
    "torchvision",
    "numpy",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "seaborn",
    "pandas",
    "Pillow"
]
with open("requirements.txt", "w") as f:
    f.write("\n".join(reqs))
print("✅ requirements.txt created.")